Cortex Agents for Microsoft Teams - Full Setup Guide
*Co-authored with CoCo*

## Prerequisites

| Requirement | Details |
|---|---|
| Snowflake Role | ACCOUNTADMIN or SECURITYADMIN |
| Azure Role | Global Administrator (ask your Azure admin) |
| Microsoft Tenant ID | Found in Azure Portal > Azure Active Directory > Properties |
| User Accounts | Each user needs both a Microsoft AND Snowflake account |
| Licenses | Microsoft Teams license (+ Copilot license if using M365 Copilot) |
| Private Link | Must be DISABLED (not supported) |

---
## Step 1: Entra ID Tenant-Wide Consent (Azure Admin Required)

**You cannot do this step** — send the following to your Azure Global Administrator:

---

> **REQUEST: Grant Entra ID consent for Snowflake Cortex Agents**
>
> Please open these two URLs in a browser (signed in as Global Admin) and click **Accept** on each dialog:
>
> **URL 1 — OAuth Resource:**
> ```
> https://login.microsoftonline.com/<YOUR-TENANT-ID>/adminconsent?client_id=5a840489-78db-4a42-8772-47be9d833efe
> ```
> → Click Accept
>
> **URL 2 — OAuth Client:**
> ```
> https://login.microsoftonline.com/<YOUR-TENANT-ID>/adminconsent?client_id=bfdfa2a2-bce5-4aee-ad3d-41ef70eb5086
> ```
> → Click Accept on BOTH dialogs (2 of 2)
>
> **Note:** If you see an error about "missing query string parameter: code" — ignore it. Consent was granted.
>
> **Verification:** In Entra admin center > Enterprise Applications, search "Snowflake Cortex Agent" — both apps should appear.

---

---
## Step 2: Create Snowflake Security Integration

Replace `<YOUR-TENANT-ID>` below with your Microsoft Tenant ID, then run the cell.

In [ ]:
%%sql -r security_integration_result
-- Step 2: Create the External OAuth Security Integration
-- IMPORTANT: Replace <YOUR-TENANT-ID> with your actual Microsoft Tenant ID

USE ROLE ACCOUNTADMIN;

CREATE OR REPLACE SECURITY INTEGRATION entra_id_cortex_agents_integration
    TYPE = EXTERNAL_OAUTH
    ENABLED = TRUE
    EXTERNAL_OAUTH_TYPE = AZURE
    EXTERNAL_OAUTH_ISSUER = 'https://login.microsoftonline.com/<YOUR-TENANT-ID>/v2.0'
    EXTERNAL_OAUTH_JWS_KEYS_URL = 'https://login.microsoftonline.com/<YOUR-TENANT-ID>/discovery/v2.0/keys'
    EXTERNAL_OAUTH_AUDIENCE_LIST = ('5a840489-78db-4a42-8772-47be9d833efe')
    EXTERNAL_OAUTH_TOKEN_USER_MAPPING_CLAIM = ('email', 'upn')
    EXTERNAL_OAUTH_SNOWFLAKE_USER_MAPPING_ATTRIBUTE = 'email_address'
    EXTERNAL_OAUTH_ANY_ROLE_MODE = 'ENABLE';

---
## Step 3: Verify User Email Mapping

The integration maps Entra ID users to Snowflake users by email. Your Snowflake user's EMAIL must exactly match your Microsoft/Entra ID email.

In [ ]:
%%sql -r user_details
-- Step 3: Check your user's email configuration
DESCRIBE USER JSINGH26;

In [ ]:
%%sql -r update_email_result
-- If your email is missing or incorrect, update it here:
-- Replace 'your.email@company.com' with your actual Microsoft email

-- ALTER USER JSINGH26 SET EMAIL = 'your.email@company.com';

---
## Step 4: Create a Non-Admin Role for Teams Bot Connection

The connecting user's default role **cannot** be ACCOUNTADMIN or SECURITYADMIN (they are blocked by default). Create a dedicated role:

In [ ]:
%%sql -r create_role_result
-- Step 4: Create a dedicated role for the Teams integration
USE ROLE ACCOUNTADMIN;

CREATE ROLE IF NOT EXISTS CORTEX_TEAMS_ROLE
    COMMENT = 'Role for Cortex Agents Teams integration';

-- Grant the role to your user
GRANT ROLE CORTEX_TEAMS_ROLE TO USER JSINGH26;

-- Grant warehouse usage
GRANT USAGE ON WAREHOUSE COMPUTE_WH TO ROLE CORTEX_TEAMS_ROLE;

In [ ]:
%%sql -r secondary_roles_result
-- Enable secondary roles so you don't have to change your default role
ALTER USER JSINGH26 SET DEFAULT_SECONDARY_ROLES = ('ALL');

---
## Step 5: Create a Cortex Agent

You need at least one Cortex Agent for Teams to use. Choose one of the options below based on your use case.

**Option A:** Agent with a Semantic View (structured data Q&A)  
**Option B:** Agent with Cortex Search (unstructured data / document search)  
**Option C:** Agent with both tools

In [ ]:
%%sql -r agent_creation_result
-- Step 5: Example Cortex Agent creation
-- Modify the semantic_view or cortex_search_service references to match your objects

-- OPTION A: Agent with Semantic View (text-to-SQL)
-- Uncomment and modify:
/*
CREATE OR REPLACE CORTEX AGENT my_teams_agent
  COMMENT = 'Agent for Microsoft Teams integration'
  FROM SPECIFICATION $$
  models:
    - name: claude-sonnet
  tools:
    - tool_spec:
        type: cortex_analyst_text_to_sql
        name: analyst
      tool_resources:
        cortex_analyst_text_to_sql:
          semantic_view: "<DATABASE>.<SCHEMA>.<YOUR_SEMANTIC_VIEW>"
  $$;
*/

-- OPTION B: Agent with Cortex Search (document/unstructured search)
-- Uncomment and modify:
/*
CREATE OR REPLACE CORTEX AGENT my_teams_agent
  COMMENT = 'Agent for Microsoft Teams integration'
  FROM SPECIFICATION $$
  models:
    - name: claude-sonnet
  tools:
    - tool_spec:
        type: cortex_search
        name: search
      tool_resources:
        cortex_search:
          cortex_search_service: "<DATABASE>.<SCHEMA>.<YOUR_SEARCH_SERVICE>"
  $$;
*/

SELECT 'Uncomment and modify one of the agent options above' AS reminder;

---
## Step 6: Grant Agent Access to the Teams Role

In [ ]:
%%sql -r grant_result
-- Step 6: Grant the Teams role access to the agent
-- Uncomment after creating your agent:

/*
GRANT USAGE ON CORTEX AGENT my_teams_agent TO ROLE CORTEX_TEAMS_ROLE;

-- Also grant access to underlying databases/schemas the agent needs
GRANT USAGE ON DATABASE <YOUR_DB> TO ROLE CORTEX_TEAMS_ROLE;
GRANT USAGE ON SCHEMA <YOUR_DB>.<YOUR_SCHEMA> TO ROLE CORTEX_TEAMS_ROLE;
*/

SELECT 'Uncomment grants after creating your agent' AS reminder;

---
## Step 7: Install & Connect in Microsoft Teams

Once your Azure admin completes Step 1:

1. **Install the app:** In Microsoft Teams, go to the App Store and search for **"Snowflake Cortex Agents"** → Click **Add**
2. **Connect your account:**
   - Click "I'm the Snowflake administrator"
   - Enter your account URL: `tnb80367.snowflakecomputing.com`
   - The wizard validates and connects
3. **Start chatting:** Ask natural language questions about your data!

### Available Commands in Teams
| Command | Description |
|---|---|
| Help | Show available commands |
| Choose agent | Switch between agents |
| Logout | Log out from current account |
| Show configured accounts | List all connected accounts |
| Clear context | Reset chat history |
| Add account | Connect another Snowflake account |

---
## Troubleshooting

| Error Code | Meaning | Fix |
|---|---|---|
| 390303 | Invalid OAuth token | Check tenant-id in security integration URLs |
| 390304 | Incorrect username | Email/UPN mismatch between Entra ID and Snowflake user |
| 390317 | Role not in token | Run: `ALTER SECURITY INTEGRATION ... SET EXTERNAL_OAUTH_ANY_ROLE_MODE = 'ENABLE'` |
| 390186 | Role not granted | User's default role is blocked — use secondary roles |

In [ ]:
%%sql -r describe_integration
-- Troubleshooting: Verify your security integration settings
-- Run this after creating the integration to confirm configuration

DESCRIBE SECURITY INTEGRATION entra_id_cortex_agents_integration;